# Sharing Args with for_context()

When a test exercises multiple endpoints for the same entity—creating an order and then fetching it, for example—an identifier like `order_id` must appear in every `respond()` call. Repeating it by hand on each call makes the intent less clear and creates multiple places to update when the value changes.

`for_context()` solves this by letting you declare a set of shared args once and have them automatically injected into every `respond()` call inside the block. Args set on the context have lower priority than explicit kwargs on individual calls, so overriding a single field for one interaction still works without affecting the others.

By completing this notebook you will have registered multiple interactions that share a common `order_id` through a context, verified the priority order by overriding one arg in a single call, and seen two independent contexts coexist on the same server instance.

## Table of Contents

- [Prerequisites](#Prerequisites)
- [1. Setup](#1-Setup)
- [2. The Repetition Problem](#2-The-Repetition-Problem)
  - [2.1 Registering Repeated Args](#21-Registering-Repeated-Args)
- [3. Sharing Args with for_context()](#3-Sharing-Args-with-for_context)
  - [3.1 Injecting Context Args](#31-Injecting-Context-Args)
- [4. Arg Resolution Priority](#4-Arg-Resolution-Priority)
  - [4.1 Explicit Kwargs Override the Context](#41-Explicit-Kwargs-Override-the-Context)
- [5. Multiple Independent Contexts](#5-Multiple-Independent-Contexts)
  - [5.1 Two Context Blocks on One Instance](#51-Two-Context-Blocks-on-One-Instance)
- [6. Conclusion](#6-Conclusion)

## Prerequisites

No environment variables are required.

Additional prerequisites:

- `mockture` must be installed in the current Python environment.
- `httpx` must be installed (`pip install httpx`).
- `configs/basic_api.openapi.yml` and `configs/basic_api.templates.yml` must be present in the same directory as this notebook.

## 1. Setup

We import `Mockture` and define a helper that constructs a fresh instance for each example. Using a helper keeps the repeated constructor arguments out of every cell.

In [ ]:
from pathlib import Path
import httpx
from mockture.server import Mockture

ROOT      = Path(".").resolve()
CONTRACT  = str(ROOT / "configs" / "basic_api.openapi.yml")
TEMPLATES = str(ROOT / "configs" / "basic_api.templates.yml")

def new_mock(**kwargs):
    return Mockture(contract_path=CONTRACT, templates_path=TEMPLATES, **kwargs)

## 2. The Repetition Problem

Consider a test that sets up a create-then-fetch sequence. Both the `create_order_success` and `get_order` templates require an `order_id`. Without a shared context, the value must be passed on every individual `respond()` call.

### 2.1 Registering Repeated Args

We register three interactions—one for `POST /orders` and two for `GET /orders/{order_id}` at different status values. The `order_id` appears on every call, making it easy to miss one if the value changes.

In [ ]:
mock = new_mock(strict=True)

# order_id repeated on every call — fragile if the value changes
mock.respond("create_order_success", order_id="ord-001", order_status="queued")
mock.respond("get_order",            order_id="ord-001", order_status="queued")
mock.respond("get_order",            order_id="ord-001", order_status="processing")

mock.start()

r1 = httpx.post(mock.url_for("/orders"),         json={"item_id": "A", "quantity": 1}, timeout=5.0)
r2 = httpx.get( mock.url_for("/orders/ord-001"),                                       timeout=5.0)

print("POST:", r1.status_code, r1.json())
print("GET: ", r2.status_code, r2.json())

mock.stop()

## 3. Sharing Args with for_context()

`for_context()` returns a `MocktureContext` object. Every `ctx.respond()` call automatically merges the context args in at a lower priority than explicit kwargs on the call itself.

- **MocktureContext**: a wrapper around a `Mockture` instance that injects a fixed set of args into every `respond()` call made through it.

### 3.1 Injecting Context Args

We rewrite the previous example using `for_context()`. The `order_id` is declared once on the context; the individual `respond()` calls only supply the args that differ between interactions.

In [ ]:
mock = new_mock(strict=True)

with mock.for_context(order_id="ord-ctx-001") as ctx:
    ctx.respond("create_order_success", order_status="queued")
    ctx.respond("get_order",            order_status="queued")
    ctx.respond("get_order",            order_status="processing")
    # order_id is injected from the context into every call above

mock.start()

r1 = httpx.post(mock.url_for("/orders"),             json={"item_id": "A", "quantity": 1}, timeout=5.0)
r2 = httpx.get( mock.url_for("/orders/ord-ctx-001"),                                       timeout=5.0)

print("POST:", r1.status_code, r1.json())
print("GET: ", r2.status_code, r2.json())

mock.stop()

## 4. Arg Resolution Priority

When Mockture resolves the final args for an interaction, it applies them in this order, from highest to lowest priority:

1. Explicit kwargs on the `respond()` call
2. Args set on the context via `for_context()`
3. Default values declared in the template

This means you can override a single arg for one interaction without affecting the others in the same context block.

### 4.1 Explicit Kwargs Override the Context

We set both `order_id` and `order_status` on the context, then override only `order_status` on the second `respond()` call. The `order_id` still comes from the context for both interactions.

In [ ]:
mock = new_mock(strict=True)

with mock.for_context(order_id="ctx-default", order_status="created") as ctx:
    ctx.respond("create_order_success")                  # uses both context values
    ctx.respond("get_order", order_status="processing")  # overrides status; order_id still from ctx

mock.start()

r_post = httpx.post(mock.url_for("/orders"),              json={"item_id": "B", "quantity": 1}, timeout=5.0)
r_get  = httpx.get( mock.url_for("/orders/ctx-default"),                                        timeout=5.0)

print("POST response:", r_post.json())
print("  order_id from context:", r_post.json()["order_id"])
print("  status  from context: ", r_post.json()["status"])
print()
print("GET response:", r_get.json())
print("  order_id still from context:  ", r_get.json()["order_id"])
print("  status overridden in respond():", r_get.json()["status"])

mock.stop()

## 5. Multiple Independent Contexts

Multiple `for_context()` blocks on the same instance register independent sets of interactions. The blocks do not interfere with each other—each one contributes its own interactions to the server's queue.

### 5.1 Two Context Blocks on One Instance

We register interactions for two separate orders—A and B—each with a different `order_id` and `order_status`. After starting the server, requests for each order receive their respective configured responses.

In [ ]:
mock = new_mock(strict=True)

with mock.for_context(order_id="ord-A") as ctx:
    ctx.respond("create_order_success", order_status="created")
    ctx.respond("get_order",            order_status="created")

with mock.for_context(order_id="ord-B") as ctx:
    ctx.respond("create_order_success", order_status="queued")
    ctx.respond("get_order",            order_status="queued")

mock.start()

httpx.post(mock.url_for("/orders"), json={"item_id": "A", "quantity": 1}, timeout=5.0)
httpx.post(mock.url_for("/orders"), json={"item_id": "B", "quantity": 1}, timeout=5.0)

r_a = httpx.get(mock.url_for("/orders/ord-A"), timeout=5.0)
r_b = httpx.get(mock.url_for("/orders/ord-B"), timeout=5.0)

print("Order A status:", r_a.json()["status"])
print("Order B status:", r_b.json()["status"])

mock.stop()

## 6. Conclusion

This notebook demonstrated `for_context()` and the arg resolution model:

- Registered multiple interactions that share a common `order_id` by declaring it once on the context rather than repeating it on every `respond()` call.
- Confirmed that an explicit kwarg on a `respond()` call overrides the context value for that arg, while other args continue to come from the context.
- Used two independent `for_context()` blocks on the same instance to register interactions for two separate entities without interference.

`for_context()` works identically inside a plugin-injected `mockture` fixture—call `mockture.for_context(...)` rather than `mock.for_context(...)`. The `for_incident()` method is a domain-specific shorthand for `for_context(incident_id=...)` and follows the same resolution rules.